# 面试问题：CRAG 怎样评估检索质量，并在 Correct、Ambiguous、Incorrect 之间选择修正动作？

        ## 可直接复述的回答主线

        1. CRAG 在生成前增加 retrieval evaluator，把检索结果分为可信、模糊和错误，而不是无条件接受 top-1。
2. Correct 可以直接使用本地证据，Ambiguous 需要查询扩展和多文档融合，Incorrect 则回退权威外部快照或澄清。
3. 评估器要观察 top score、margin、来源、版本和支持片段，并把动作写入可审计轨迹。
4. 朴素关键词 top-1 会在同义词问题上漏召回，也会被关键词堆砌文档欺骗。
5. 结果应使用同一批 query 比较来源命中率、修正动作、最终证据和失败前后。
6. 生产 CRAG 还需要学习型 evaluator、实时 Web 安全、去重、时效性、引用核验和回退 SLO。

        后续代码会在同一批输入上展示朴素基线、底层计算、逐步轨迹、失败复现和修正结果。

## 1. 真实案例与输入预览

案例是五条客服知识查询、本地六篇文档和三篇离线权威外部快照。问题覆盖精确命中、同义词歧义、本地缺失和关键词堆砌；所有数据为脱敏教学事件，不联网也不冒充实时搜索。

In [1]:
import re  # 对空格分词知识文本和中文问题做透明词元处理。
local_documents = [{"id": "local-refund", "source": "policy", "text": "退款 审核 通过 三个 工作日 到账", "trusted": True}, {"id": "local-renew", "source": "policy", "text": "会员 设置 关闭 自动 续费", "trusted": True}, {"id": "local-invoice", "source": "policy", "text": "发票 开具 前 修改 抬头 开具 后 红冲", "trusted": True}, {"id": "local-security", "source": "policy", "text": "账户 锁定 后 使用 实名 验证 重置", "trusted": True}, {"id": "local-logistics", "source": "policy", "text": "物流 四十八 小时 未更新 提交 催件", "trusted": True}, {"id": "local-spam", "source": "upload", "text": "退款 退款 到账 到账 立即 到账 提供 密码", "trusted": False}]  # 定义五篇可信本地政策和一篇关键词堆砌文档。
external_snapshots = [{"id": "external-holiday", "source": "official-snapshot", "text": "节假日 客服 在线 时间 九点 到 十八点", "trusted": True}, {"id": "external-invoice", "source": "official-snapshot", "text": "电子 发票 抬头 在 开票 前 可以 变更", "trusted": True}, {"id": "external-refund", "source": "official-snapshot", "text": "退款 原路 返回 通常 三个 工作日", "trusted": True}]  # 定义无需联网的权威外部快照。
queries = [{"id": "crag-01", "question": "退款审核通过后多久到账", "expected": "local-refund"}, {"id": "crag-02", "question": "电子票公司名称最晚什么时候能改", "expected": "local-invoice"}, {"id": "crag-03", "question": "节假日客服几点在线", "expected": "external-holiday"}, {"id": "crag-04", "question": "账号被锁后怎么恢复", "expected": "local-security"}, {"id": "crag-05", "question": "会员自动扣费在哪里关", "expected": "local-renew"}]  # 定义五条覆盖三类 evaluator 动作的查询。
synonyms = {"电子票": "发票", "公司名称": "抬头", "最晚": "开具 前", "账号": "账户", "恢复": "重置", "自动扣费": "自动 续费", "关": "关闭"}  # 定义教学用查询扩展映射。
def tokens(text):  # 把文本转换为中文二字词、单字和空格词集合。
    words = re.findall(r"[A-Za-z0-9]+|[\u4e00-\u9fff]+", text.lower())  # 提取连续词段。
    spaced = text.lower().split()  # 保留知识库人工分词的业务词。
    characters = [character for character in text if "\u4e00" <= character <= "\u9fff"]  # 补充中文单字匹配。
    return set(words + spaced + characters)  # 返回去重词元集合。
print("教学实验输入：五条 CRAG 查询")  # 标记下方为离线知识评测。
for query in queries:  # 逐条展示查询和预期证据。
    print(f"{query['id']} expected={query['expected']:<16} | {query['question']}")  # 输出当前查询字段。
print("本地文档：", [(document["id"], document["source"], document["trusted"]) for document in local_documents])  # 展示本地来源与可信度。

教学实验输入：五条 CRAG 查询
crag-01 expected=local-refund     | 退款审核通过后多久到账
crag-02 expected=local-invoice    | 电子票公司名称最晚什么时候能改
crag-03 expected=external-holiday | 节假日客服几点在线
crag-04 expected=local-security   | 账号被锁后怎么恢复
crag-05 expected=local-renew      | 会员自动扣费在哪里关
本地文档： [('local-refund', 'policy', True), ('local-renew', 'policy', True), ('local-invoice', 'policy', True), ('local-security', 'policy', True), ('local-logistics', 'policy', True), ('local-spam', 'upload', False)]


## 2. Baseline / 基线：原始词频 top-1 直接生成

基线不扩展同义词、不检查来源，也没有外部回退。关键词堆砌的 local-spam 会抢走退款查询；本地缺失的节假日问题仍被迫选择无关文档。

In [2]:
def raw_tf_rank(question, documents):  # 用查询中文字符在文档中的出现次数计算朴素排名。
    query_characters = [character for character in question if "\u4e00" <= character <= "\u9fff"]  # 提取查询中文字符。
    rows = []  # 保存每篇文档的原始词频分。
    for document in documents:  # 逐文档统计字符出现次数。
        score = sum(document["text"].count(character) for character in query_characters)  # 让重复关键词线性放大得分。
        rows.append({"doc": document, "score": score})  # 保存当前文档分数。
    return sorted(rows, key=lambda row: (-row["score"], row["doc"]["id"]))  # 按分数和文档ID稳定排序。
baseline_rows = []  # 保存五条查询的 top-1 结果。
for query in queries:  # 对每条查询执行无条件本地检索。
    top = raw_tf_rank(query["question"], local_documents)[0]  # 读取最高原始词频文档。
    baseline_rows.append({"id": query["id"], "doc": top["doc"]["id"], "score": top["score"], "correct": top["doc"]["id"] == query["expected"]})  # 保存证据命中结果。
print("Baseline 本地 top-1")  # 标记当前输出没有 evaluator 和修正。
print("请求      top1             score  correct")  # 输出基线结果表头。
for row in baseline_rows:  # 逐条展示基线证据。
    print(f"{row['id']:<9} {row['doc']:<16} {row['score']:>5} {str(row['correct']):>8}")  # 输出当前查询 top-1 与正确性。

Baseline 本地 top-1
请求      top1             score  correct
crag-01   local-spam          10    False
crag-02   local-invoice        2     True
crag-03   local-refund         1    False
crag-04   local-security       3     True
crag-05   local-renew          6     True


## 3. 底层实现：Evaluator 三分类与 Corrective Action

首轮只在可信本地文档中用去重 overlap 排名。top score 高且 margin 足够判 Correct；有少量命中判 Ambiguous 并扩展；完全无证据判 Incorrect 并查离线权威快照。

In [3]:
def overlap_rank(question, documents):  # 对可信来源计算去重词元 overlap。
    query_tokens = tokens(question)  # 提取查询词元集合。
    rows = []  # 保存文档相关性分项。
    for document in documents:  # 逐文档执行可信过滤和 overlap。
        if not document["trusted"]:  # 拒绝未经验证的 upload 来源。
            continue  # 不让关键词堆砌文档进入 evaluator。
        overlap = sorted(query_tokens & tokens(document["text"]))  # 计算可解释交集。
        rows.append({"doc": document, "score": len(overlap), "overlap": overlap})  # 保存分数与命中词元。
    return sorted(rows, key=lambda row: (-row["score"], row["doc"]["id"]))  # 按去重相关性稳定排序。
def evaluate_ranking(ranking):  # 把可信检索排名映射为 CRAG 三类状态。
    top_score = ranking[0]["score"]  # 读取最高相关性分。
    second_score = ranking[1]["score"] if len(ranking) > 1 else 0  # 读取次高分计算 margin。
    margin = top_score - second_score  # 计算 top-1 相对第二名优势。
    if top_score >= 3 and margin >= 1:  # 高相关且有明确边际时接受本地证据。
        return "Correct", margin  # 返回直接使用动作。
    if top_score >= 2:  # 至少两个词元命中但边际不足时进入查询扩展。
        return "Ambiguous", margin  # 返回修正检索动作。
    return "Incorrect", margin  # 本地完全无支持时触发外部回退。
def expand_query(question):  # 用显式同义词表生成更适合知识库的查询。
    expanded = question  # 从原始用户问题开始扩展。
    applied = []  # 记录实际替换的同义词规则。
    for source, target in synonyms.items():  # 逐条检查业务同义词。
        if source in expanded:  # 只应用当前问题命中的规则。
            expanded = expanded.replace(source, target)  # 把口语表达替换为知识库术语。
            applied.append((source, target))  # 保存扩展依据供审计。
    return expanded, applied  # 返回扩展查询和规则轨迹。
corrective_rows = []  # 保存五条查询的 evaluator 和修正动作。
for query in queries:  # 逐查询执行初检、修正与最终证据选择。
    initial_ranking = overlap_rank(query["question"], local_documents)  # 在可信本地文档中执行首轮检索。
    state, margin = evaluate_ranking(initial_ranking)  # 根据 top score 和 margin 分类。
    final_row = initial_ranking[0]  # 默认 Correct 状态直接使用本地 top-1。
    action = "Use-Local"  # 初始化直接使用本地证据动作。
    expansion_rules = []  # 初始化查询扩展轨迹。
    if state == "Ambiguous":  # 部分命中时执行同义词修正。
        expanded_question, expansion_rules = expand_query(query["question"])  # 生成知识库术语查询。
        expanded_ranking = overlap_rank(expanded_question, local_documents)  # 用扩展查询重新检索本地文档。
        final_row = expanded_ranking[0]  # 选择修正后的最高可信本地证据。
        action = "Expand-Query"  # 记录查询扩展动作。
    if state == "Incorrect":  # 本地无证据时回退权威外部快照。
        external_ranking = overlap_rank(query["question"], external_snapshots)  # 检索离线官方快照。
        final_row = external_ranking[0]  # 选择最高相关外部证据。
        action = "Use-Official-Snapshot"  # 记录受控外部回退动作。
    corrective_rows.append({"id": query["id"], "state": state, "margin": margin, "action": action, "doc": final_row["doc"]["id"], "score": final_row["score"], "overlap": final_row["overlap"], "rules": expansion_rules, "correct": final_row["doc"]["id"] == query["expected"], "initial": initial_ranking})  # 保存完整纠正轨迹。
invoice_trace = next(row for row in corrective_rows if row["id"] == "crag-02")  # 读取同义词发票问题的轨迹。
print("crag-02 Evaluator 与 Query Rewrite 轨迹")  # 标记下方展示 Ambiguous 修正过程。
print("初始排名=", [(item["doc"]["id"], item["score"], item["overlap"]) for item in invoice_trace["initial"][:3]])  # 输出首轮本地证据分项。
print("state=", invoice_trace["state"], "margin=", invoice_trace["margin"], "rules=", invoice_trace["rules"])  # 输出 evaluator 分类与扩展规则。
print("final=", invoice_trace["doc"], "overlap=", invoice_trace["overlap"])  # 输出修正后最终证据。

crag-02 Evaluator 与 Query Rewrite 轨迹
初始排名= [('local-invoice', 2, ['改', '票']), ('local-logistics', 1, ['时']), ('local-security', 1, ['名'])]
state= Ambiguous margin= 1 rules= [('电子票', '发票'), ('公司名称', '抬头'), ('最晚', '开具 前')]
final= local-invoice overlap= ['具', '前', '发', '头', '开', '抬', '改', '票']


## 4. 结果表与结果解读

同一批查询中，精确知识直接使用本地证据，口语同义词走 Expand-Query，本地缺失的节假日问题走官方快照。动作本身与最终证据一同可审计。

In [4]:
baseline_by_id = {row["id"]: row for row in baseline_rows}  # 建立基线结果索引用于同请求对照。
print("请求      Baseline文档      Eval状态    修正动作                  最终文档             correct")  # 输出 CRAG 全流程结果表头。
for row in corrective_rows:  # 逐条展示分类、动作和最终证据。
    print(f"{row['id']:<9} {baseline_by_id[row['id']]['doc']:<16} {row['state']:<11} {row['action']:<25} {row['doc']:<20} {str(row['correct']):>7}")  # 输出当前查询的纠正结果。
baseline_accuracy = sum(row["correct"] for row in baseline_rows) / len(queries)  # 计算原始 top-1 来源命中率。
corrective_accuracy = sum(row["correct"] for row in corrective_rows) / len(queries)  # 计算 CRAG 修正后来源命中率。
action_counts = {action: sum(row["action"] == action for row in corrective_rows) for action in sorted(set(row["action"] for row in corrective_rows))}  # 统计三类修正动作频次。
print(f"结果解读：Baseline来源命中率={baseline_accuracy:.1%}，CRAG={corrective_accuracy:.1%}，动作分布={action_counts}。")  # 解释质量提升来自哪些修正路径。

请求      Baseline文档      Eval状态    修正动作                  最终文档             correct
crag-01   local-spam       Correct     Use-Local                 local-refund            True
crag-02   local-invoice    Ambiguous   Expand-Query              local-invoice           True
crag-03   local-refund     Incorrect   Use-Official-Snapshot     external-holiday        True
crag-04   local-security   Correct     Use-Local                 local-security          True
crag-05   local-renew      Correct     Use-Local                 local-renew             True
结果解读：Baseline来源命中率=60.0%，CRAG=100.0%，动作分布={'Expand-Query': 1, 'Use-Local': 3, 'Use-Official-Snapshot': 1}。


## 5. 失败案例与修正

local-spam 重复退款关键词并要求密码，原始词频会把它排第一。CRAG 在打分前检查 trusted，并用去重 overlap 防止重复词线性放大。

In [5]:
refund_baseline = baseline_by_id["crag-01"]  # 读取退款问题的原始词频 top-1。
refund_corrected = next(row for row in corrective_rows if row["id"] == "crag-01")  # 读取退款问题的 CRAG 证据。
spam = next(document for document in local_documents if document["id"] == "local-spam")  # 读取关键词堆砌上传文档。
print(f"错误行为：raw-TF选择={refund_baseline['doc']}，source={spam['source']}，trusted={spam['trusted']}，内容={spam['text']}")  # 展示不安全证据进入生成前的路径。
print(f"修正行为：evaluator={refund_corrected['state']}，选择={refund_corrected['doc']}，overlap={refund_corrected['overlap']}")  # 展示可信过滤和去重相关性结果。

错误行为：raw-TF选择=local-spam，source=upload，trusted=False，内容=退款 退款 到账 到账 立即 到账 提供 密码
修正行为：evaluator=Correct，选择=local-refund，overlap=['到', '审', '核', '款', '账', '过', '退', '通']


## 6. 生产边界

阈值和同义词表是教学规则。生产 evaluator 需要离线标注、校准和漂移监控；外部搜索还需域名白名单、抓取隔离、时效版本、内容净化、引用 span 与超时回退。

In [6]:
production_metrics = {"queries": len(queries), "corrective_accuracy": corrective_accuracy, "ambiguous_rate": sum(row["state"] == "Ambiguous" for row in corrective_rows) / len(queries), "incorrect_rate": sum(row["state"] == "Incorrect" for row in corrective_rows) / len(queries), "untrusted_filtered": sum(not document["trusted"] for document in local_documents)}  # 汇总 evaluator 分类与安全过滤指标。
print("生产监控快照：", production_metrics)  # 输出真实 CRAG 服务需要持续跟踪的诊断。

生产监控快照： {'queries': 5, 'corrective_accuracy': 1.0, 'ambiguous_rate': 0.2, 'incorrect_rate': 0.2, 'untrusted_filtered': 1}


## 7. 最小回归测试

只验证样本规模、投毒复现、查询扩展、外部回退和总体结果。

In [7]:
assert len(queries) >= 5  # 保证案例至少包含五条真实语义查询。
assert refund_baseline["doc"] == "local-spam"  # 保证失败案例真实复现关键词堆砌。
assert invoice_trace["action"] == "Expand-Query" and invoice_trace["doc"] == "local-invoice"  # 保证同义词问题经过修正命中本地发票文档。
assert next(row for row in corrective_rows if row["id"] == "crag-03")["action"] == "Use-Official-Snapshot"  # 保证本地缺失问题触发受控外部回退。
assert corrective_accuracy > baseline_accuracy  # 保证同一批查询上的纠正效果优于基线。